**Data Acquisition and Initialization**

I will be using the COVID-19 Deaths Dataset from Kaggle, specifically the us-counties.csv file. This dataset contains over 1.7 million records of COVID-19 cases and deaths across various U.S. counties. In this section, we load the data into a Pandas DataFrame to prepare for processing.

In [17]:
import pandas as pd

# Load the file from the data folder
df = pd.read_csv('../data/us-counties.csv')

# Verify the row count
print(f"Successfully loaded {len(df):,} rows.")

Successfully loaded 1,774,204 rows.


**Sequential Data Processing (Baseline)**

The sequential approach processes the data on a single CPU core. We perform three main operations:

Filter: Select counties with more than 1,000 cases.

Aggregate: Sum the total deaths per state.

Sort: Identify the top 10 states with the highest death counts.

We measure the execution time to establish a performance baseline.

In [18]:
import time

def sequential_analytics(data):
    # 1. Filter: Hotspots (Counties with more than 1,000 cases)
    filtered = data[data['cases'] > 1000]
    
    # 2. Aggregate: Total deaths grouped by State
    state_deaths = filtered.groupby('state')['deaths'].sum()
    
    # 3. Sort: Top 10 states by total death count
    top_states = state_deaths.sort_values(ascending=False).head(10)
    
    return top_states

# --- Execution and Timing ---
start_time = time.time()

# Run the sequential version
result_seq = sequential_analytics(df)

end_time = time.time()
sequential_duration = end_time - start_time

print(f"Sequential Execution Time: {sequential_duration:.4f} seconds")
print("\n--- Top 10 States (Sequential Results) ---")
print(result_seq)

Sequential Execution Time: 0.0031 seconds

--- Top 10 States (Sequential Results) ---
state
New York        31931
California      29721
Texas           10948
Arizona          8096
Illinois         7454
Rhode Island     1866
Nevada           1751
Florida           952
Connecticut       837
Michigan          770
Name: deaths, dtype: int64


**Parallel Data Processing (MapReduce)**

To improve processing for larger datasets, we implement a MapReduce pattern using the multiprocessing library:

Map: The dataset is split into chunks and distributed across multiple CPU cores.

Process: Each core runs the parallel_worker function independently.

Reduce: The results from all cores are combined back into a single result set.

Note: On Windows, the worker function must be defined in an external worker.py file to avoid recursion issues during the "spawn" process.

In [19]:
import multiprocessing as mp
import time
import pandas as pd
from worker import parallel_worker

if __name__ == '__main__':
    num_processes = 2
    
    # --- PROPER PANDAS SPLIT ---
    # This keeps the column names (headers) attached to each chunk
    chunk_size = len(df) // num_processes
    chunks = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]
    
    start_time = time.time()
    
    # Create the pool
    with mp.Pool(processes=num_processes) as pool:
        results = pool.map(parallel_worker, chunks)
    
    # Combine (Reduce) and Sort
    final_result = pd.concat(results).groupby(level=0).sum().sort_values(ascending=False).head(10)
    
    parallel_duration = time.time() - start_time
    print(f"Parallel Execution Time: {parallel_duration:.4f} seconds")
    print(final_result)

Parallel Execution Time: 0.7326 seconds
state
New York        31931
California      29721
Texas           10948
Arizona          8096
Illinois         7454
Rhode Island     1866
Nevada           1751
Florida           952
Connecticut       837
Michigan          770
Name: deaths, dtype: int64
